# Olist Brazilian E-Commerce Data Cleaning

This notebook implements the data-cleaning decisions identified during the
data-understanding stage.

The objective is to prepare reliable and analysis-ready datasets while
preserving valid transactional records and maintaining the relational
structure of the original Olist data.

The cleaning process includes:

- Data type conversion
- ZIP code identifier normalization
- Duplicate removal
- Geolocation coordinate filtering and ZIP-level aggregation
- Product measurement cleaning
- Preservation of valid payment, review, and order records
- Validation of cleaned datasets before export

All transformations are applied to copies of the raw datasets so that the
original source data remains unchanged.

## 1. Environment Setup

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

## 2. Load Raw Datasets

Load the original Olist datasets from the raw-data directory.

Copies of the source datasets will be created before transformations are
applied so that the raw data remains unchanged.

In [2]:
data_path = Path("../data/raw")

customers = pd.read_csv(
    data_path / "olist_customers_dataset.csv"
)

geolocation = pd.read_csv(
    data_path / "olist_geolocation_dataset.csv"
)

order_items = pd.read_csv(
    data_path / "olist_order_items_dataset.csv"
)

payments = pd.read_csv(
    data_path / "olist_order_payments_dataset.csv"
)

reviews = pd.read_csv(
    data_path / "olist_order_reviews_dataset.csv"
)

orders = pd.read_csv(
    data_path / "olist_orders_dataset.csv"
)

products = pd.read_csv(
    data_path / "olist_products_dataset.csv"
)

sellers = pd.read_csv(
    data_path / "olist_sellers_dataset.csv"
)

category_translation = pd.read_csv(
    data_path / "product_category_name_translation.csv"
)

### 2.1 Create Working Copies

Create independent working copies of each dataset before applying cleaning
transformations.

In [3]:
customers_clean = customers.copy()
geolocation_clean = geolocation.copy()
order_items_clean = order_items.copy()
payments_clean = payments.copy()
reviews_clean = reviews.copy()
orders_clean = orders.copy()
products_clean = products.copy()
sellers_clean = sellers.copy()
category_translation_clean = category_translation.copy()

## 3. Data Type Cleaning

### 3.1 ZIP Code Identifier Conversion

ZIP code prefix fields are stored as integers in the raw datasets.

Because ZIP code prefixes are geographic identifiers rather than numeric
quantities, they are converted to five-character strings. Leading zeros are
restored where necessary to preserve a consistent identifier format.

The following fields are standardized:

- `customers.customer_zip_code_prefix`
- `geolocation.geolocation_zip_code_prefix`
- `sellers.seller_zip_code_prefix`

In [4]:
customers_clean["customer_zip_code_prefix"] = (
    customers_clean["customer_zip_code_prefix"]
    .astype(str)
    .str.zfill(5)
)

geolocation_clean["geolocation_zip_code_prefix"] = (
    geolocation_clean["geolocation_zip_code_prefix"]
    .astype(str)
    .str.zfill(5)
)

sellers_clean["seller_zip_code_prefix"] = (
    sellers_clean["seller_zip_code_prefix"]
    .astype(str)
    .str.zfill(5)
)

zip_dtype_check = pd.DataFrame({
    "dataset": [
        "customers",
        "geolocation",
        "sellers"
    ],
    "column": [
        "customer_zip_code_prefix",
        "geolocation_zip_code_prefix",
        "seller_zip_code_prefix"
    ],
    "dtype": [
        customers_clean["customer_zip_code_prefix"].dtype,
        geolocation_clean["geolocation_zip_code_prefix"].dtype,
        sellers_clean["seller_zip_code_prefix"].dtype
    ]
})

zip_dtype_check

zip_format_check = pd.DataFrame({
    "dataset": [
        "customers",
        "geolocation",
        "sellers"
    ],
    "minimum_length": [
        customers_clean["customer_zip_code_prefix"].str.len().min(),
        geolocation_clean["geolocation_zip_code_prefix"].str.len().min(),
        sellers_clean["seller_zip_code_prefix"].str.len().min()
    ],
    "maximum_length": [
        customers_clean["customer_zip_code_prefix"].str.len().max(),
        geolocation_clean["geolocation_zip_code_prefix"].str.len().max(),
        sellers_clean["seller_zip_code_prefix"].str.len().max()
    ],
    "invalid_length_records": [
        customers_clean["customer_zip_code_prefix"].str.len().ne(5).sum(),
        geolocation_clean["geolocation_zip_code_prefix"].str.len().ne(5).sum(),
        sellers_clean["seller_zip_code_prefix"].str.len().ne(5).sum()
    ]
})

zip_format_check

,dataset,minimum_length,maximum_length,invalid_length_records
0,customers,5,5,0
1,geolocation,5,5,0
2,sellers,5,5,0


In [5]:
customers_clean.loc[
    customers_clean["customer_zip_code_prefix"].str.startswith("0"),
    ["customer_zip_code_prefix"]
].head()

,customer_zip_code_prefix
1,09790
2,01151
3,08775
6,04534
13,05704


### 3.2 Datetime Conversion

Several order, review, and shipping-related date fields are stored as strings
in the raw datasets.

These fields are converted to datetime values so that they can be used for
time-based analysis, duration calculations, and delivery-performance metrics.

In [6]:
datetime_columns = {
    "order_items": [
        "shipping_limit_date"
    ],
    "reviews": [
        "review_creation_date",
        "review_answer_timestamp"
    ],
    "orders": [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
}

for column in datetime_columns["order_items"]:
    order_items_clean[column] = pd.to_datetime(
        order_items_clean[column],
        errors="coerce"
    )

for column in datetime_columns["reviews"]:
    reviews_clean[column] = pd.to_datetime(
        reviews_clean[column],
        errors="coerce"
    )

for column in datetime_columns["orders"]:
    orders_clean[column] = pd.to_datetime(
        orders_clean[column],
        errors="coerce"
    )

In [7]:
datetime_dtype_check = pd.DataFrame({
    "dataset": [
        "order_items",
        "reviews",
        "reviews",
        "orders",
        "orders",
        "orders",
        "orders",
        "orders"
    ],
    "column": [
        "shipping_limit_date",
        "review_creation_date",
        "review_answer_timestamp",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ],
    "dtype": [
        order_items_clean["shipping_limit_date"].dtype,
        reviews_clean["review_creation_date"].dtype,
        reviews_clean["review_answer_timestamp"].dtype,
        orders_clean["order_purchase_timestamp"].dtype,
        orders_clean["order_approved_at"].dtype,
        orders_clean["order_delivered_carrier_date"].dtype,
        orders_clean["order_delivered_customer_date"].dtype,
        orders_clean["order_estimated_delivery_date"].dtype
    ]
})

datetime_dtype_check

,dataset,column,dtype
0,order_items,shipping_limit_date,datetime64[us]
1,reviews,review_creation_date,datetime64[us]
2,reviews,review_answer_timestamp,datetime64[us]
3,orders,order_purchase_timestamp,datetime64[us]
4,orders,order_approved_at,datetime64[us]
5,orders,order_delivered_carrier_date,datetime64[us]
6,orders,order_delivered_customer_date,datetime64[us]
7,orders,order_estimated_delivery_date,datetime64[us]


#### Datetime Conversion Validation

Because invalid date strings are converted to `NaT` when using
`errors="coerce"`, the number of missing values after conversion is checked
against the original datasets.

This ensures that datetime conversion does not introduce additional missing
values.

In [8]:
datetime_missing_check = pd.DataFrame({
    "dataset": [
        "order_items",
        "reviews",
        "reviews",
        "orders",
        "orders",
        "orders",
        "orders",
        "orders"
    ],
    "column": [
        "shipping_limit_date",
        "review_creation_date",
        "review_answer_timestamp",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ],
    "missing_before": [
        order_items["shipping_limit_date"].isna().sum(),
        reviews["review_creation_date"].isna().sum(),
        reviews["review_answer_timestamp"].isna().sum(),
        orders["order_purchase_timestamp"].isna().sum(),
        orders["order_approved_at"].isna().sum(),
        orders["order_delivered_carrier_date"].isna().sum(),
        orders["order_delivered_customer_date"].isna().sum(),
        orders["order_estimated_delivery_date"].isna().sum()
    ],
    "missing_after": [
        order_items_clean["shipping_limit_date"].isna().sum(),
        reviews_clean["review_creation_date"].isna().sum(),
        reviews_clean["review_answer_timestamp"].isna().sum(),
        orders_clean["order_purchase_timestamp"].isna().sum(),
        orders_clean["order_approved_at"].isna().sum(),
        orders_clean["order_delivered_carrier_date"].isna().sum(),
        orders_clean["order_delivered_customer_date"].isna().sum(),
        orders_clean["order_estimated_delivery_date"].isna().sum()
    ]
})

datetime_missing_check["new_missing"] = (
    datetime_missing_check["missing_after"]
    - datetime_missing_check["missing_before"]
)

datetime_missing_check

,dataset,column,missing_before,missing_after,new_missing
0,order_items,shipping_limit_date,0,0,0
1,reviews,review_creation_date,0,0,0
2,reviews,review_answer_timestamp,0,0,0
3,orders,order_purchase_timestamp,0,0,0
4,orders,order_approved_at,160,160,0
5,orders,order_delivered_carrier_date,1783,1783,0
6,orders,order_delivered_customer_date,2965,2965,0
7,orders,order_estimated_delivery_date,0,0,0


#### Datetime Conversion Findings

All eight date and timestamp fields were successfully converted to datetime
values.

The number of missing values remained unchanged before and after conversion,
with zero newly introduced missing values. This confirms that no valid date
strings were lost during the conversion process.

Existing missing lifecycle timestamps in the `orders` dataset are preserved
for downstream analysis.

## 4. Geolocation Cleaning

The raw geolocation dataset contains exact duplicate observations, multiple
geographic observations for individual ZIP code prefixes, and a small number
of clearly unusual coordinates.

The cleaning process will:

- Remove exact duplicate geographic observations
- Exclude coordinates outside the broad geographic sanity-check bounds
- Preserve valid observations for each ZIP code prefix
- Create a ZIP-code-level geographic representation for downstream joins

### 4.1 Remove Exact Duplicate Observations

Exact duplicate geographic observations do not provide additional
information and may unnecessarily increase the number of records associated
with a ZIP code prefix.

These duplicate observations are removed before coordinate validation and
ZIP-level aggregation.

In [9]:
geo_rows_before = len(geolocation_clean)

geolocation_clean = (
    geolocation_clean
    .drop_duplicates()
    .copy()
)

geo_rows_after = len(geolocation_clean)

print(
    "Rows before duplicate removal:",
    geo_rows_before
)

print(
    "Rows after duplicate removal:",
    geo_rows_after
)

print(
    "Exact duplicates removed:",
    geo_rows_before - geo_rows_after
)

print(
    "Remaining exact duplicates:",
    geolocation_clean.duplicated().sum()
)

Rows before duplicate removal: 1000163
Rows after duplicate removal: 738332
Exact duplicates removed: 261831
Remaining exact duplicates: 0


#### Duplicate Removal Findings

A total of 261,831 exact duplicate geographic observations were removed from
the `geolocation` dataset.

The number of records decreased from 1,000,163 to 738,332, with no exact
duplicate records remaining after the cleaning step.

Only exact duplicate observations were removed at this stage. Multiple
distinct geographic observations associated with the same ZIP code prefix
are preserved for subsequent coordinate validation and ZIP-level
aggregation.

### 4.2 Invalid Coordinate Removal

The data-understanding stage identified a small number of geolocation
observations outside broad geographic sanity-check bounds for Brazil.

After exact duplicate removal, the coordinate validation is applied again
to the remaining observations because some previously identified anomalies
may themselves have been duplicate records.

The same broad sanity-check bounds are used:

- Latitude: -35 to 6
- Longitude: -75 to -30

Observations outside either bound are excluded before ZIP-level geographic
aggregation. These thresholds are intended to identify clearly unusual
coordinates rather than define the exact national borders of Brazil.

In [10]:
invalid_coordinate_mask = (
    ~geolocation_clean["geolocation_lat"].between(-35, 6)
    |
    ~geolocation_clean["geolocation_lng"].between(-75, -30)
)

invalid_coordinates = geolocation_clean[
    invalid_coordinate_mask
].copy()

print(
    "Invalid coordinate observations:",
    len(invalid_coordinates)
)

print(
    "ZIP code prefixes affected:",
    invalid_coordinates[
        "geolocation_zip_code_prefix"
    ].nunique()
)

invalid_coordinates[
    [
        "geolocation_zip_code_prefix",
        "geolocation_lat",
        "geolocation_lng",
        "geolocation_city",
        "geolocation_state"
    ]
].sort_values(
    ["geolocation_lat", "geolocation_lng"]
)

Invalid coordinate observations: 25
ZIP code prefixes affected: 20


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
992584,98780,-36.605374,-64.283946,santa rosa,RS
993075,98780,-36.603837,-64.287433,santa rosa,RS
965687,95130,14.585073,121.105394,santa lucia do piai,RS
538557,29654,21.657547,-101.466766,santo antonio do canaa,ES
585242,35179,25.995203,-98.078544,santana do paraíso,MG
585260,35179,25.995245,-98.078533,santana do paraiso,MG
387565,18243,28.008978,-15.536867,bom retiro da esperanca,SP
538512,29654,29.409252,-98.484121,santo antônio do canaã,ES
698466,47310,38.268205,-7.803886,santana do sobrado,BA
695377,45936,38.323939,-6.775035,itabatan,BA


In [11]:
geo_rows_before_coordinate_removal = len(
    geolocation_clean
)

geolocation_clean = geolocation_clean[
    ~invalid_coordinate_mask
].copy()

geo_rows_after_coordinate_removal = len(
    geolocation_clean
)

print(
    "Rows before coordinate removal:",
    geo_rows_before_coordinate_removal
)

print(
    "Rows after coordinate removal:",
    geo_rows_after_coordinate_removal
)

print(
    "Invalid coordinates removed:",
    geo_rows_before_coordinate_removal
    - geo_rows_after_coordinate_removal
)

Rows before coordinate removal: 738332
Rows after coordinate removal: 738307
Invalid coordinates removed: 25


In [12]:
remaining_invalid_coordinates = (
    ~geolocation_clean["geolocation_lat"].between(-35, 6)
    |
    ~geolocation_clean["geolocation_lng"].between(-75, -30)
).sum()

print(
    "Remaining invalid coordinates:",
    remaining_invalid_coordinates
)

Remaining invalid coordinates: 0


#### Coordinate Cleaning Findings

After exact duplicate removal, 25 unique geographic observations remained
outside the broad coordinate sanity-check bounds.

These observations affected 20 ZIP code prefixes and were excluded before
ZIP-level geographic aggregation.

The number of geolocation records decreased from 738,332 to 738,307, with
no observations remaining outside the defined sanity-check bounds.

Only the invalid geographic observations were removed. The affected ZIP code
prefixes themselves were preserved because other valid geographic
observations may still exist for those ZIP prefixes.

### 4.3 Validate Affected ZIP Code Coverage

Before aggregating the geolocation data to ZIP-code level, the ZIP code
prefixes affected by coordinate removal are checked to determine whether
valid geographic observations remain.

This validation ensures that removing invalid coordinate observations does
not unintentionally eliminate an entire ZIP code prefix from the geographic
lookup data.

In [13]:
affected_zip_codes = set(
    invalid_coordinates["geolocation_zip_code_prefix"]
)

remaining_affected_zip_counts = (
    geolocation_clean[
        geolocation_clean["geolocation_zip_code_prefix"].isin(
            affected_zip_codes
        )
    ]
    .groupby("geolocation_zip_code_prefix")
    .size()
    .reset_index(name="valid_observations_remaining")
)

affected_zip_validation = pd.DataFrame({
    "geolocation_zip_code_prefix": sorted(
        affected_zip_codes
    )
}).merge(
    remaining_affected_zip_counts,
    on="geolocation_zip_code_prefix",
    how="left"
)

affected_zip_validation[
    "valid_observations_remaining"
] = (
    affected_zip_validation[
        "valid_observations_remaining"
    ]
    .fillna(0)
    .astype(int)
)

affected_zip_validation

,geolocation_zip_code_prefix,valid_observations_remaining
0,18243,0
1,28155,4
2,28165,3
3,28333,5
4,28595,8
5,29654,2
6,35179,65
7,45936,31
8,46560,1
9,47310,4


In [14]:
zip_codes_with_valid_observations = (
    affected_zip_validation[
        "valid_observations_remaining"
    ] > 0
).sum()

zip_codes_without_valid_observations = (
    affected_zip_validation[
        "valid_observations_remaining"
    ] == 0
).sum()

print(
    "Affected ZIP prefixes:",
    len(affected_zip_validation)
)

print(
    "ZIP prefixes with valid observations remaining:",
    zip_codes_with_valid_observations
)

print(
    "ZIP prefixes with no valid observations remaining:",
    zip_codes_without_valid_observations
)

Affected ZIP prefixes: 20
ZIP prefixes with valid observations remaining: 16
ZIP prefixes with no valid observations remaining: 4


#### Affected ZIP Code Coverage Findings

Of the 20 ZIP code prefixes affected by invalid coordinate removal, 16
retained at least one valid geographic observation.

Four ZIP code prefixes (`18243`, `78131`, `83252`, and `95130`) had no valid
observations remaining after the coordinate sanity check.

No replacement coordinates are imputed because the available geolocation
data does not provide sufficient evidence for a reliable estimate. These ZIP
code prefixes may therefore have missing geographic coordinates in downstream
analysis.

### 4.4 ZIP-Level Geolocation Aggregation

The cleaned geolocation dataset still contains multiple valid observations
for many ZIP code prefixes.

To create a geographic lookup table suitable for joining with customers and
sellers, the observations are aggregated to one record per ZIP code prefix.

The median latitude and longitude are used as representative coordinates
because the median is less sensitive to remaining coordinate variation than
the arithmetic mean.

The number of valid observations contributing to each ZIP-level coordinate
is also retained for transparency.

In [15]:
geolocation_zip = (
    geolocation_clean
    .groupby(
        "geolocation_zip_code_prefix",
        as_index=False
    )
    .agg(
        geolocation_lat=("geolocation_lat", "median"),
        geolocation_lng=("geolocation_lng", "median"),
        observation_count=("geolocation_lat", "size")
    )
)

geolocation_zip.head()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,observation_count
0,01001,-23.549951,-46.634027,11
1,01002,-23.548228,-46.635247,6
2,01003,-23.548977,-46.635313,11
3,01004,-23.549550,-46.634771,14
4,01005,-23.549763,-46.636100,13


In [16]:
print(
    "Clean geolocation observations:",
    len(geolocation_clean)
)

print(
    "ZIP-level records:",
    len(geolocation_zip)
)

print(
    "Unique ZIP prefixes:",
    geolocation_zip[
        "geolocation_zip_code_prefix"
    ].nunique()
)

print(
    "Duplicate ZIP prefixes:",
    geolocation_zip[
        "geolocation_zip_code_prefix"
    ].duplicated().sum()
)

print(
    "Missing latitude:",
    geolocation_zip[
        "geolocation_lat"
    ].isna().sum()
)

print(
    "Missing longitude:",
    geolocation_zip[
        "geolocation_lng"
    ].isna().sum()
)

Clean geolocation observations: 738307
ZIP-level records: 19011
Unique ZIP prefixes: 19011
Duplicate ZIP prefixes: 0
Missing latitude: 0
Missing longitude: 0


#### ZIP-Level Aggregation Findings

After duplicate removal and coordinate validation, 738,307 valid geographic
observations were aggregated into 19,011 ZIP-level records.

Each ZIP code prefix is represented by the median latitude and longitude of
its valid observations, together with the number of observations contributing
to the representative coordinate.

The resulting lookup table contains one unique record per ZIP code prefix,
with no duplicate ZIP prefixes or missing representative coordinates.

Compared with the 19,015 ZIP code prefixes observed in the raw geolocation
data, four ZIP prefixes are absent because all of their available coordinate
observations failed the geographic sanity check.

### 4.5 Customer and Seller ZIP Coverage

The ZIP-level geolocation lookup is validated against the ZIP code prefixes
used in the customer and seller datasets.

This step measures how many customer and seller records can be matched to a
valid representative geographic coordinate and identifies records that may
have missing geographic information in downstream analysis.

In [17]:
customer_geo_check = customers_clean[
    ["customer_id", "customer_zip_code_prefix"]
].merge(
    geolocation_zip[
        [
            "geolocation_zip_code_prefix",
            "geolocation_lat",
            "geolocation_lng"
        ]
    ],
    left_on="customer_zip_code_prefix",
    right_on="geolocation_zip_code_prefix",
    how="left"
)

customer_geo_matched = (
    customer_geo_check["geolocation_lat"]
    .notna()
    .sum()
)

customer_geo_unmatched = (
    customer_geo_check["geolocation_lat"]
    .isna()
    .sum()
)

print("Customer records:", len(customer_geo_check))
print("Matched customer records:", customer_geo_matched)
print("Unmatched customer records:", customer_geo_unmatched)
print(
    "Customer match rate:",
    f"{customer_geo_matched / len(customer_geo_check):.2%}"
)

Customer records: 99441
Matched customer records: 99162
Unmatched customer records: 279
Customer match rate: 99.72%


In [18]:
seller_geo_check = sellers_clean[
    ["seller_id", "seller_zip_code_prefix"]
].merge(
    geolocation_zip[
        [
            "geolocation_zip_code_prefix",
            "geolocation_lat",
            "geolocation_lng"
        ]
    ],
    left_on="seller_zip_code_prefix",
    right_on="geolocation_zip_code_prefix",
    how="left"
)

seller_geo_matched = (
    seller_geo_check["geolocation_lat"]
    .notna()
    .sum()
)

seller_geo_unmatched = (
    seller_geo_check["geolocation_lat"]
    .isna()
    .sum()
)

print("Seller records:", len(seller_geo_check))
print("Matched seller records:", seller_geo_matched)
print("Unmatched seller records:", seller_geo_unmatched)
print(
    "Seller match rate:",
    f"{seller_geo_matched / len(seller_geo_check):.2%}"
)

Seller records: 3095
Matched seller records: 3088
Unmatched seller records: 7
Seller match rate: 99.77%


In [19]:
unmatched_customer_zips = (
    customer_geo_check.loc[
        customer_geo_check["geolocation_lat"].isna(),
        "customer_zip_code_prefix"
    ]
    .value_counts()
    .rename_axis("zip_code_prefix")
    .reset_index(name="customer_records")
)

unmatched_seller_zips = (
    seller_geo_check.loc[
        seller_geo_check["geolocation_lat"].isna(),
        "seller_zip_code_prefix"
    ]
    .value_counts()
    .rename_axis("zip_code_prefix")
    .reset_index(name="seller_records")
)

print(
    "Unique unmatched customer ZIP prefixes:",
    len(unmatched_customer_zips)
)

print(
    "Unique unmatched seller ZIP prefixes:",
    len(unmatched_seller_zips)
)

Unique unmatched customer ZIP prefixes: 158
Unique unmatched seller ZIP prefixes: 7


In [20]:
unmatched_customer_zips.head(10)

,zip_code_prefix,customer_records
0,70686,15
1,72005,13
2,71919,10
3,73255,7
4,72300,6
5,73369,5
6,71884,5
7,72002,5
8,73401,5
9,71676,4


In [21]:
unmatched_seller_zips.head(10)

,zip_code_prefix,seller_records
0,82040,1
1,91901,1
2,72580,1
3,02285,1
4,07412,1
5,71551,1
6,37708,1


In [22]:
raw_geo_zips = set(
    geolocation[
        "geolocation_zip_code_prefix"
    ]
    .astype(str)
)

removed_geo_zips = (
    affected_zip_validation.loc[
        affected_zip_validation[
            "valid_observations_remaining"
        ] == 0,
        "geolocation_zip_code_prefix"
    ]
    .tolist()
)

print(
    "Unmatched customer ZIPs present in raw geolocation:",
    unmatched_customer_zips[
        "zip_code_prefix"
    ].isin(raw_geo_zips).sum()
)

print(
    "Unmatched seller ZIPs present in raw geolocation:",
    unmatched_seller_zips[
        "zip_code_prefix"
    ].isin(raw_geo_zips).sum()
)

print(
    "ZIP prefixes removed entirely by coordinate cleaning:",
    removed_geo_zips
)

Unmatched customer ZIPs present in raw geolocation: 1
Unmatched seller ZIPs present in raw geolocation: 0
ZIP prefixes removed entirely by coordinate cleaning: ['18243', '78131', '83252', '95130']


#### Geographic Coverage Findings

The cleaned ZIP-level geolocation lookup provides geographic coordinates for
99.72% of customer records and 99.77% of seller records.

Among the 99,441 customer records, 99,162 were successfully matched to a
valid ZIP-level coordinate, while 279 records across 158 ZIP code prefixes
remained unmatched.

Among the 3,095 seller records, 3,088 were successfully matched, while seven
records across seven ZIP code prefixes remained unmatched.

Only one of the unmatched customer ZIP code prefixes was present in the raw
geolocation dataset, while none of the unmatched seller ZIP code prefixes
were present. This indicates that the remaining geographic coverage gaps are
primarily due to limitations in the original geolocation dataset rather than
the cleaning process.

Customer and seller records without geographic matches are retained because
their transactional information remains valid. They will only lack
coordinate-based information in analyses that require geographic data.

## 5. Product Data Cleaning

The product dataset contains missing descriptive attributes and a small
number of zero product-weight values identified during the data-understanding
stage.

The cleaning strategy preserves valid product and transaction records while
avoiding unsupported numerical imputation.

The main product-cleaning actions include:

- Convert zero product weights to missing values
- Preserve existing missing descriptive and physical attributes
- Avoid imputing numerical product measurements without sufficient evidence
- Validate that product records and identifiers remain intact after cleaning

### 5.1 Zero Product Weight Cleaning

Four products were identified with a recorded weight of zero grams.

These products are associated with valid transactions and contain other
product information, so the product records should not be removed. Because a
zero physical weight is not considered a meaningful product measurement, the
zero values are converted to missing values.

No replacement weight is imputed because the available data does not provide
sufficient evidence for a reliable estimate.

In [23]:
zero_weight_before = (
    products_clean["product_weight_g"] == 0
).sum()

missing_weight_before = (
    products_clean["product_weight_g"].isna().sum()
)

print(
    "Zero product weights before cleaning:",
    zero_weight_before
)

print(
    "Missing product weights before cleaning:",
    missing_weight_before
)

Zero product weights before cleaning: 4
Missing product weights before cleaning: 2


In [24]:
products_clean.loc[
    products_clean["product_weight_g"] == 0,
    "product_weight_g"
] = np.nan

In [25]:
zero_weight_after = (
    products_clean["product_weight_g"] == 0
).sum()

missing_weight_after = (
    products_clean["product_weight_g"].isna().sum()
)

print(
    "Zero product weights after cleaning:",
    zero_weight_after
)

print(
    "Missing product weights after cleaning:",
    missing_weight_after
)

print(
    "New missing weights introduced:",
    missing_weight_after - missing_weight_before
)

Zero product weights after cleaning: 0
Missing product weights after cleaning: 6
New missing weights introduced: 4


#### Zero Product Weight Cleaning Findings

Four products with a recorded weight of zero grams were reclassified as
missing values because zero does not represent a meaningful physical product
weight.

The number of missing product weights increased from 2 to 6, reflecting the
four invalid zero-weight values that were converted to missing values.

No product records were removed, and no replacement weights were imputed.
This preserves valid transactional relationships while avoiding unsupported
assumptions about the products' actual weights.

### 5.2 Missing Product Attribute Validation

Missing product attributes identified during the data-understanding stage are
reviewed after the zero-weight correction.

Missing descriptive and physical attributes are preserved rather than
imputed because the available data does not provide sufficient evidence for
reliable replacement values.

This validation confirms the remaining missing-value pattern before the
cleaned product dataset is used in downstream analysis.

In [26]:
product_missing_summary = (
    products_clean
    .isna()
    .sum()
    .to_frame("missing_count")
)

product_missing_summary["missing_percentage"] = (
    product_missing_summary["missing_count"]
    / len(products_clean)
    * 100
).round(2)

product_missing_summary = (
    product_missing_summary[
        product_missing_summary["missing_count"] > 0
    ]
    .sort_values(
        "missing_count",
        ascending=False
    )
)

product_missing_summary

,missing_count,missing_percentage
product_category_name,610,1.85
product_name_lenght,610,1.85
product_description_lenght,610,1.85
product_photos_qty,610,1.85
product_weight_g,6,0.02
product_length_cm,2,0.01
product_height_cm,2,0.01
product_width_cm,2,0.01


In [27]:
product_integrity_check = pd.DataFrame({
    "metric": [
        "Raw product records",
        "Clean product records",
        "Raw unique product IDs",
        "Clean unique product IDs",
        "Duplicate product IDs after cleaning"
    ],
    "value": [
        len(products),
        len(products_clean),
        products["product_id"].nunique(),
        products_clean["product_id"].nunique(),
        products_clean["product_id"].duplicated().sum()
    ]
})

product_integrity_check

,metric,value
0,Raw product records,32951
1,Clean product records,32951
2,Raw unique product IDs,32951
3,Clean unique product IDs,32951
4,Duplicate product IDs after cleaning,0


#### Missing Product Attribute Findings

The product dataset retains 32,951 records and 32,951 unique product IDs
after cleaning, with no duplicate product identifiers introduced.

A total of 610 products (1.85%) have missing category and descriptive
attributes, including product category, name length, description length, and
photo quantity.

After reclassifying the four zero-weight values, six products (0.02%) have
missing weight information. Two products (0.01%) also have missing length,
height, and width measurements.

These missing attributes are preserved rather than numerically imputed
because the available data does not provide sufficient evidence for reliable
replacement values. Product records are retained so that their valid
transactional relationships remain available for downstream analysis.

### 5.3 Missing Product Category Handling

Products with missing category information are retained because they may
still be associated with valid transactions.

Rather than assigning these products to an inferred category, missing
`product_category_name` values are labeled as `unknown`. This provides an
explicit category for downstream aggregation and reporting while avoiding
unsupported assumptions about the products' actual classifications.

Other missing descriptive and physical product attributes remain unchanged.

In [28]:
missing_category_before = (
    products_clean["product_category_name"]
    .isna()
    .sum()
)

products_clean["product_category_name"] = (
    products_clean["product_category_name"]
    .fillna("unknown")
)

missing_category_after = (
    products_clean["product_category_name"]
    .isna()
    .sum()
)

unknown_category_count = (
    products_clean["product_category_name"]
    .eq("unknown")
    .sum()
)

print(
    "Missing categories before cleaning:",
    missing_category_before
)

print(
    "Missing categories after cleaning:",
    missing_category_after
)

print(
    "Products labeled as unknown:",
    unknown_category_count
)

Missing categories before cleaning: 610
Missing categories after cleaning: 0
Products labeled as unknown: 610


#### Product Category Cleaning Findings

All 610 products with missing category information were retained and assigned
the explicit category label `unknown`.

After this transformation, no missing values remain in
`product_category_name`. The `unknown` label preserves these products in
category-level analysis without making unsupported assumptions about their
actual classifications.

Other missing descriptive and physical product attributes remain unchanged.

## 6. Transactional Data Validation

The orders, payments, and reviews datasets contain several unusual or missing
values identified during the data-understanding stage.

These records are not automatically removed because many of the observed
patterns are structurally valid or do not provide sufficient evidence of
data-entry errors.

This section validates the cleaned transactional datasets and documents the
records that are intentionally preserved for downstream analysis.

### 6.1 Order Lifecycle Timestamp Validation

Missing order lifecycle timestamps are preserved because their interpretation
depends on the order status and the analytical metric being calculated.

For example, canceled or unavailable orders may legitimately lack approval,
carrier-delivery, or customer-delivery timestamps.

Delivered orders with missing lifecycle timestamps are also retained. They
will only be excluded from calculations that specifically require the missing
timestamp.

In [29]:
order_timestamp_columns = [
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date"
]

order_timestamp_missing_by_status = (
    orders_clean
    .groupby("order_status")[order_timestamp_columns]
    .agg(lambda x: x.isna().sum())
)

order_timestamp_missing_by_status

,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date
order_status,,,
approved,0,2,2
canceled,141,550,619
created,5,5,5
delivered,14,2,8
invoiced,0,314,314
processing,0,301,301
shipped,0,0,1107
unavailable,0,609,609


In [30]:
delivered_orders = orders_clean[
    orders_clean["order_status"] == "delivered"
]

delivered_timestamp_missing = pd.DataFrame({
    "column": order_timestamp_columns,
    "missing_count": [
        delivered_orders[column].isna().sum()
        for column in order_timestamp_columns
    ]
})

delivered_timestamp_missing

,column,missing_count
0,order_approved_at,14
1,order_delivered_carrier_date,2
2,order_delivered_customer_date,8


#### Order Lifecycle Timestamp Findings

Missing lifecycle timestamps vary substantially by order status and are
largely consistent with the progression of the order lifecycle.

Orders with statuses such as `created`, `canceled`, `invoiced`, `processing`,
`shipped`, and `unavailable` frequently lack later-stage timestamps because
those lifecycle events may not have occurred.

Among delivered orders, 14 records are missing `order_approved_at`, two are
missing `order_delivered_carrier_date`, and eight are missing
`order_delivered_customer_date`.

These records are retained without timestamp imputation. They will only be
excluded from downstream metrics that specifically require the missing
timestamp.

### 6.2 Payment Anomaly Validation

The payment dataset contains a small number of zero-value payments and
zero-installment records identified during the data-understanding stage.

These records are preserved because there is insufficient evidence to
determine reliable replacement values or to classify the records as invalid
transactions.

The anomalies are validated and documented rather than automatically removed
or imputed.

In [31]:
payment_anomalies = payments_clean[
    (payments_clean["payment_value"] == 0)
    |
    (payments_clean["payment_installments"] == 0)
].copy()

payment_anomalies.sort_values(
    [
        "payment_value",
        "payment_installments"
    ]
)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
19922,8bcbe01d44d147f901cd3192671144db,4,voucher,1,0.00
36822,fa65dad1b0e818e3ccc5cb0e39231352,14,voucher,1,0.00
43744,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.00
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.00
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.00
62674,45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.00
77885,fa65dad1b0e818e3ccc5cb0e39231352,13,voucher,1,0.00
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.00
100766,b23878b3e8eb4d25a158f57d96331b18,4,voucher,1,0.00
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69


In [32]:
payment_anomaly_summary = pd.DataFrame({
    "anomaly": [
        "Zero payment value",
        "Zero payment installments",
        "Total anomaly records"
    ],
    "record_count": [
        (payments_clean["payment_value"] == 0).sum(),
        (payments_clean["payment_installments"] == 0).sum(),
        len(payment_anomalies)
    ]
})

payment_anomaly_summary

,anomaly,record_count
0,Zero payment value,9
1,Zero payment installments,2
2,Total anomaly records,11


In [33]:
payment_integrity_check = pd.DataFrame({
    "metric": [
        "Raw payment records",
        "Clean payment records",
        "Raw unique orders",
        "Clean unique orders",
        "Duplicate composite keys after cleaning"
    ],
    "value": [
        len(payments),
        len(payments_clean),
        payments["order_id"].nunique(),
        payments_clean["order_id"].nunique(),
        payments_clean[
            ["order_id", "payment_sequential"]
        ].duplicated().sum()
    ]
})

payment_integrity_check

,metric,value
0,Raw payment records,103886
1,Clean payment records,103886
2,Raw unique orders,99440
3,Clean unique orders,99440
4,Duplicate composite keys after cleaning,0


#### Payment Anomaly Findings

The payment dataset contains nine records with a payment value of zero and
two credit-card records with zero installments, resulting in 11 identified
payment anomalies.

The zero-value payments consist of voucher and `not_defined` payment types,
while the two zero-installment credit-card records have positive payment
values.

Because the available data does not provide sufficient evidence for reliable
replacement values or for classifying these records as invalid transactions,
all 11 records are retained without modification.

The cleaned payment dataset preserves all 103,886 payment records and 99,440
unique orders. The composite key (`order_id`, `payment_sequential`) remains
unique, with no duplicate composite keys introduced during cleaning.

### 6.3 Review Data Validation

The review dataset contains substantial missing values in the optional review
comment fields and may contain multiple review records associated with the
same order or review identifier.

Missing comment titles and messages are preserved because customers are not
required to provide written feedback.

Review records are also preserved at their original grain. Neither
`review_id` nor `order_id` is treated as a standalone primary key; the
combination of `review_id` and `order_id` is used to validate record
uniqueness.

In [34]:
review_missing_summary = pd.DataFrame({
    "column": [
        "review_score",
        "review_comment_title",
        "review_comment_message"
    ],
    "missing_count": [
        reviews_clean["review_score"].isna().sum(),
        reviews_clean["review_comment_title"].isna().sum(),
        reviews_clean["review_comment_message"].isna().sum()
    ]
})

review_missing_summary["missing_percentage"] = (
    review_missing_summary["missing_count"]
    / len(reviews_clean)
    * 100
).round(2)

review_missing_summary

,column,missing_count,missing_percentage
0,review_score,0,0.00
1,review_comment_title,87656,88.34
2,review_comment_message,58247,58.70


In [35]:
review_integrity_check = pd.DataFrame({
    "metric": [
        "Raw review records",
        "Clean review records",
        "Unique review IDs",
        "Unique order IDs",
        "Duplicate review IDs",
        "Duplicate order IDs",
        "Duplicate composite keys"
    ],
    "value": [
        len(reviews),
        len(reviews_clean),
        reviews_clean["review_id"].nunique(),
        reviews_clean["order_id"].nunique(),
        reviews_clean["review_id"].duplicated().sum(),
        reviews_clean["order_id"].duplicated().sum(),
        reviews_clean[
            ["review_id", "order_id"]
        ].duplicated().sum()
    ]
})

review_integrity_check

,metric,value
0,Raw review records,99224
1,Clean review records,99224
2,Unique review IDs,98410
3,Unique order IDs,98673
4,Duplicate review IDs,814
5,Duplicate order IDs,551
6,Duplicate composite keys,0


In [36]:
review_score_validation = pd.DataFrame({
    "metric": [
        "Minimum review score",
        "Maximum review score",
        "Invalid review scores"
    ],
    "value": [
        reviews_clean["review_score"].min(),
        reviews_clean["review_score"].max(),
        (~reviews_clean["review_score"].between(1, 5)).sum()
    ]
})

review_score_validation

,metric,value
0,Minimum review score,1
1,Maximum review score,5
2,Invalid review scores,0


#### Review Data Validation Findings

The cleaned review dataset preserves all 99,224 original review records.

Written review content is optional and remains substantially incomplete:
88.34% of review titles and 58.70% of review messages are missing. These
missing values are preserved because the absence of written feedback does not
invalidate the associated review score.

The `review_id` and `order_id` fields are not individually unique, with 814
duplicate review IDs and 551 duplicate order IDs. However, the composite key
(`review_id`, `order_id`) contains no duplicates, confirming that the
original review-record grain remains intact.

Review scores range from 1 to 5, with no missing or invalid score values.
Therefore, no review records are removed or imputed during cleaning.

### 6.4 Order Item Value Validation

Order item price and freight values are validated to confirm the anomalies
identified during the data-understanding stage.

Zero freight values are not automatically treated as invalid because they
may represent legitimate free-shipping transactions. Records are preserved
unless there is sufficient evidence that the underlying transaction is
invalid.

In [37]:
order_item_value_validation = pd.DataFrame({
    "metric": [
        "Zero price records",
        "Zero freight records",
        "Negative price records",
        "Negative freight records"
    ],
    "record_count": [
        (order_items_clean["price"] == 0).sum(),
        (order_items_clean["freight_value"] == 0).sum(),
        (order_items_clean["price"] < 0).sum(),
        (order_items_clean["freight_value"] < 0).sum()
    ]
})

order_item_value_validation

,metric,record_count
0,Zero price records,0
1,Zero freight records,383
2,Negative price records,0
3,Negative freight records,0


#### Order Item Value Findings

No zero or negative product prices were identified in the order item dataset.

A total of 383 order item records have a freight value of zero. These records
are retained because zero freight may represent legitimate free-shipping
transactions and there is insufficient evidence to classify them as invalid.

No negative freight values were identified. Therefore, no order item records
are removed or modified based on price or freight values.

### 6.5 Transactional Dataset Integrity

A final record-count and key-integrity check is performed across the core
transactional datasets to confirm that the cleaning process has preserved
their original analytical grain.

No transactional records are expected to be removed during this stage.

In [38]:
transaction_integrity = pd.DataFrame({
    "dataset": [
        "orders",
        "order_items",
        "payments",
        "reviews"
    ],
    "raw_records": [
        len(orders),
        len(order_items),
        len(payments),
        len(reviews)
    ],
    "clean_records": [
        len(orders_clean),
        len(order_items_clean),
        len(payments_clean),
        len(reviews_clean)
    ]
})

transaction_integrity["records_removed"] = (
    transaction_integrity["raw_records"]
    - transaction_integrity["clean_records"]
)

transaction_integrity

,dataset,raw_records,clean_records,records_removed
0,orders,99441,99441,0
1,order_items,112650,112650,0
2,payments,103886,103886,0
3,reviews,99224,99224,0


In [39]:
transaction_key_validation = pd.DataFrame({
    "dataset": [
        "orders",
        "order_items",
        "payments",
        "reviews"
    ],
    "duplicate_key_records": [
        orders_clean["order_id"].duplicated().sum(),

        order_items_clean[
            ["order_id", "order_item_id"]
        ].duplicated().sum(),

        payments_clean[
            ["order_id", "payment_sequential"]
        ].duplicated().sum(),

        reviews_clean[
            ["review_id", "order_id"]
        ].duplicated().sum()
    ]
})

transaction_key_validation

,dataset,duplicate_key_records
0,orders,0
1,order_items,0
2,payments,0
3,reviews,0


#### Transactional Integrity Findings

All records in the four core transactional datasets were preserved during
cleaning.

The cleaned datasets contain 99,441 orders, 112,650 order items, 103,886
payment records, and 99,224 review records, with no records removed.

Key-integrity validation also confirms that no duplicate primary or composite
keys are present in the cleaned transactional datasets.

This confirms that the cleaning process preserved the original analytical
grain of the transactional data while retaining unusual but potentially valid
records for downstream analysis.

## 7. Analysis-Ready Dataset Preparation

The cleaned datasets are prepared for downstream Python, SQL, and Power BI
analysis while preserving their original relational structure.

A single fully flattened master table is not created because the Olist
datasets operate at different analytical grains. In particular, orders may
contain multiple items, payments, and review records.

Directly joining these one-to-many relationships into a single item-level
table could create row multiplication and lead to duplicated revenue or
transaction counts.

Instead, cleaned relational datasets and supporting lookup tables are
prepared for downstream analysis.

### 7.1 Product Category Translation

The product category translation table is joined to the cleaned product
dataset to provide English category names for downstream reporting.

Products previously labeled as `unknown` remain explicitly categorized as
unknown rather than receiving an inferred translation.

In [40]:
products_analysis = products_clean.merge(
    category_translation_clean,
    on="product_category_name",
    how="left"
)

products_analysis[
    "product_category_name_english"
] = (
    products_analysis[
        "product_category_name_english"
    ]
    .fillna("unknown")
)

products_analysis.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0,perfumery
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0,art
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0,sports_leisure
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0,baby
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0,housewares


In [41]:
product_translation_validation = pd.DataFrame({
    "metric": [
        "Clean product records",
        "Analysis product records",
        "Unique product IDs",
        "Duplicate product IDs",
        "Missing English categories",
        "Unknown English categories"
    ],
    "value": [
        len(products_clean),
        len(products_analysis),
        products_analysis["product_id"].nunique(),
        products_analysis["product_id"].duplicated().sum(),
        products_analysis[
            "product_category_name_english"
        ].isna().sum(),
        products_analysis[
            "product_category_name_english"
        ].eq("unknown").sum()
    ]
})

product_translation_validation

,metric,value
0,Clean product records,32951
1,Analysis product records,32951
2,Unique product IDs,32951
3,Duplicate product IDs,0
4,Missing English categories,0
5,Unknown English categories,623


#### Translation Coverage Investigation

The translated product dataset contains 623 products labeled with an
`unknown` English category, which is 13 more than the 610 products whose
original category was missing.

The additional unmatched records are investigated to determine which
non-missing Portuguese product categories are not covered by the translation
lookup table.

In [42]:
untranslated_products = products_analysis[
    (products_analysis["product_category_name"] != "unknown")
    &
    (products_analysis["product_category_name_english"] == "unknown")
].copy()

print(
    "Products with non-missing but untranslated categories:",
    len(untranslated_products)
)

print(
    "Unique untranslated categories:",
    untranslated_products[
        "product_category_name"
    ].nunique()
)

Products with non-missing but untranslated categories: 13
Unique untranslated categories: 2


In [43]:
untranslated_category_summary = (
    untranslated_products[
        "product_category_name"
    ]
    .value_counts()
    .rename_axis("product_category_name")
    .reset_index(name="product_count")
)

untranslated_category_summary

,product_category_name,product_count
0,portateis_cozinha_e_preparadores_de_alimentos,10
1,pc_gamer,3


#### Product Category Translation Findings

The product category translation was successfully merged without changing
the product-level grain. The resulting dataset contains 32,951 records and
32,951 unique product IDs, with no duplicate product identifiers introduced.

All missing English category values were explicitly labeled as `unknown`.
A total of 623 products received this label.

Of these, 610 products already had missing original category information.
The remaining 13 products belonged to two Portuguese categories that are not
covered by the provided translation lookup table:
`portateis_cozinha_e_preparadores_de_alimentos` (10 products) and
`pc_gamer` (3 products).

No manual translations are introduced for these categories. They remain
labeled as `unknown` in the English category field to preserve a
source-supported and reproducible transformation process.

### 7.2 Customer Geographic Enrichment

The cleaned ZIP-level geolocation lookup is joined to the customer dataset to
provide representative latitude and longitude coordinates for geographic
analysis.

A left join is used so that all customer records are preserved, including
customers whose ZIP code prefixes are not covered by the cleaned geolocation
lookup.

The ZIP-level observation count is also retained to indicate how many valid
geolocation observations contributed to each representative coordinate.

In [44]:
customers_analysis = customers_clean.merge(
    geolocation_zip,
    left_on="customer_zip_code_prefix",
    right_on="geolocation_zip_code_prefix",
    how="left"
)

customers_analysis.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,observation_count
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,14409,-20.502307,-47.396740,126.0
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,09790,sao bernardo do campo,SP,09790,-23.730435,-46.541474,125.0
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,01151,sao paulo,SP,01151,-23.531294,-46.656980,34.0
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,08775,mogi das cruzes,SP,08775,-23.499025,-46.183436,83.0
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP,13056,-22.974331,-47.142173,131.0


In [45]:
customer_geo_validation = pd.DataFrame({
    "metric": [
        "Clean customer records",
        "Analysis customer records",
        "Unique customer IDs",
        "Duplicate customer IDs",
        "Customers with coordinates",
        "Customers without coordinates",
        "Geographic coverage (%)"
    ],
    "value": [
        len(customers_clean),
        len(customers_analysis),
        customers_analysis["customer_id"].nunique(),
        customers_analysis["customer_id"].duplicated().sum(),
        customers_analysis["geolocation_lat"].notna().sum(),
        customers_analysis["geolocation_lat"].isna().sum(),
        round(
            customers_analysis["geolocation_lat"].notna().mean() * 100,
            2
        )
    ]
})

customer_geo_validation

,metric,value
0,Clean customer records,99441.00
1,Analysis customer records,99441.00
2,Unique customer IDs,99441.00
3,Duplicate customer IDs,0.00
4,Customers with coordinates,99162.00
5,Customers without coordinates,279.00
6,Geographic coverage (%),99.72


#### Customer Geographic Enrichment Findings

The ZIP-level geolocation lookup was successfully joined to the customer
dataset without changing the customer-level grain.

The resulting analysis dataset contains all 99,441 customer records and
99,441 unique customer IDs, with no duplicate customer identifiers introduced
by the geographic enrichment.

Representative coordinates are available for 99,162 customer records,
resulting in geographic coverage of 99.72%. The remaining 279 customer
records do not have a matching coordinate in the cleaned ZIP-level
geolocation lookup.

These unmatched customers are retained because their transactional and
customer information remains valid. They will only have missing coordinates
in analyses that require geographic information.

### 7.3 Seller Geographic Enrichment

The cleaned ZIP-level geolocation lookup is joined to the seller dataset to
provide representative latitude and longitude coordinates for geographic
analysis.

A left join is used to preserve all seller records, including sellers whose
ZIP code prefixes are not covered by the cleaned geolocation lookup.

The ZIP-level observation count is retained to indicate how many valid
geolocation observations contributed to each representative coordinate.

In [46]:
sellers_analysis = sellers_clean.merge(
    geolocation_zip,
    left_on="seller_zip_code_prefix",
    right_on="geolocation_zip_code_prefix",
    how="left"
)

sellers_analysis.head()

,seller_id,seller_zip_code_prefix,seller_city,seller_state,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,observation_count
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP,13023,-22.893863,-47.062006,58.0
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP,13844,-22.382869,-46.947992,91.0
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ,20031,-22.910174,-43.176775,98.0
3,c0f3eea2e14555b6faeea3dd58c1b1c3,04195,sao paulo,SP,04195,-23.656991,-46.612490,60.0
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP,12914,-22.963763,-46.534676,108.0


In [47]:
seller_geo_validation = pd.DataFrame({
    "metric": [
        "Clean seller records",
        "Analysis seller records",
        "Unique seller IDs",
        "Duplicate seller IDs",
        "Sellers with coordinates",
        "Sellers without coordinates",
        "Geographic coverage (%)"
    ],
    "value": [
        len(sellers_clean),
        len(sellers_analysis),
        sellers_analysis["seller_id"].nunique(),
        sellers_analysis["seller_id"].duplicated().sum(),
        sellers_analysis["geolocation_lat"].notna().sum(),
        sellers_analysis["geolocation_lat"].isna().sum(),
        round(
            sellers_analysis["geolocation_lat"].notna().mean() * 100,
            2
        )
    ]
})

seller_geo_validation

,metric,value
0,Clean seller records,3095.00
1,Analysis seller records,3095.00
2,Unique seller IDs,3095.00
3,Duplicate seller IDs,0.00
4,Sellers with coordinates,3088.00
5,Sellers without coordinates,7.00
6,Geographic coverage (%),99.77


#### Seller Geographic Enrichment Findings

The ZIP-level geolocation lookup was successfully joined to the seller
dataset without changing the seller-level grain.

The resulting analysis dataset contains all 3,095 seller records and 3,095
unique seller IDs, with no duplicate seller identifiers introduced by the
geographic enrichment.

Representative coordinates are available for 3,088 seller records, resulting
in geographic coverage of 99.77%. The remaining seven seller records do not
have a matching coordinate in the cleaned ZIP-level geolocation lookup.

These unmatched sellers are retained because their seller and transactional
information remains valid. They will only have missing coordinates in
analyses that require geographic information.

### 7.4 Final Analysis-Ready Table Structure

The enriched customer, seller, and product tables are finalized before
export.

Redundant join-key columns introduced during geographic enrichment are
removed, and geographic fields are renamed to clearly identify whether the
coordinates belong to a customer or seller.

The transactional datasets remain at their original cleaned grain and are
not flattened into a single master table.

In [48]:
customers_analysis = (
    customers_analysis
    .drop(columns=["geolocation_zip_code_prefix"])
    .rename(columns={
        "geolocation_lat": "customer_lat",
        "geolocation_lng": "customer_lng",
        "observation_count": "customer_geo_observation_count"
    })
)

customers_analysis.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,customer_lat,customer_lng,customer_geo_observation_count
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,-20.502307,-47.396740,126.0
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,09790,sao bernardo do campo,SP,-23.730435,-46.541474,125.0
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,01151,sao paulo,SP,-23.531294,-46.656980,34.0
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,08775,mogi das cruzes,SP,-23.499025,-46.183436,83.0
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP,-22.974331,-47.142173,131.0


In [49]:
sellers_analysis = (
    sellers_analysis
    .drop(columns=["geolocation_zip_code_prefix"])
    .rename(columns={
        "geolocation_lat": "seller_lat",
        "geolocation_lng": "seller_lng",
        "observation_count": "seller_geo_observation_count"
    })
)

sellers_analysis.head()

,seller_id,seller_zip_code_prefix,seller_city,seller_state,seller_lat,seller_lng,seller_geo_observation_count
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP,-22.893863,-47.062006,58.0
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP,-22.382869,-46.947992,91.0
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ,-22.910174,-43.176775,98.0
3,c0f3eea2e14555b6faeea3dd58c1b1c3,04195,sao paulo,SP,-23.656991,-46.612490,60.0
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP,-22.963763,-46.534676,108.0


In [50]:
analysis_table_summary = pd.DataFrame({
    "dataset": [
        "customers_analysis",
        "sellers_analysis",
        "products_analysis",
        "geolocation_zip",
        "orders_clean",
        "order_items_clean",
        "payments_clean",
        "reviews_clean"
    ],
    "rows": [
        len(customers_analysis),
        len(sellers_analysis),
        len(products_analysis),
        len(geolocation_zip),
        len(orders_clean),
        len(order_items_clean),
        len(payments_clean),
        len(reviews_clean)
    ],
    "columns": [
        customers_analysis.shape[1],
        sellers_analysis.shape[1],
        products_analysis.shape[1],
        geolocation_zip.shape[1],
        orders_clean.shape[1],
        order_items_clean.shape[1],
        payments_clean.shape[1],
        reviews_clean.shape[1]
    ]
})

analysis_table_summary

,dataset,rows,columns
0,customers_analysis,99441,8
1,sellers_analysis,3095,7
2,products_analysis,32951,10
3,geolocation_zip,19011,4
4,orders_clean,99441,8
5,order_items_clean,112650,7
6,payments_clean,103886,5
7,reviews_clean,99224,7


#### Final Analysis-Ready Table Findings

The final analysis-ready structure preserves the relational grain of the
original Olist datasets while incorporating the required cleaning and
enrichment transformations.

The customer and seller tables include representative geographic coordinates
without introducing duplicate entity records. The product table includes
English category labels while preserving one record per product.

The orders, order items, payments, and reviews datasets remain at their
original transactional grains rather than being flattened into a single
master table. This structure reduces the risk of row multiplication and
double-counting when working with one-to-many relationships.

A ZIP-level geolocation lookup is also retained as a separate supporting
table for geographic analysis.

## 8. Final Validation and Export

Before exporting the processed datasets, a final validation is performed to
confirm that the cleaned and analysis-ready tables preserve their intended
record counts, key structures, and required analytical fields.

The validated datasets will then be exported to the `data/processed`
directory for use in downstream Python, SQL, and Power BI analysis.

### 8.1 Final Key Validation

Primary and composite keys are validated one final time across the
analysis-ready datasets before export.

In [51]:
final_key_validation = pd.DataFrame({
    "dataset": [
        "customers_analysis",
        "sellers_analysis",
        "products_analysis",
        "geolocation_zip",
        "orders_clean",
        "order_items_clean",
        "payments_clean",
        "reviews_clean"
    ],
    "duplicate_key_records": [
        customers_analysis["customer_id"].duplicated().sum(),
        sellers_analysis["seller_id"].duplicated().sum(),
        products_analysis["product_id"].duplicated().sum(),
        geolocation_zip[
            "geolocation_zip_code_prefix"
        ].duplicated().sum(),
        orders_clean["order_id"].duplicated().sum(),
        order_items_clean[
            ["order_id", "order_item_id"]
        ].duplicated().sum(),
        payments_clean[
            ["order_id", "payment_sequential"]
        ].duplicated().sum(),
        reviews_clean[
            ["review_id", "order_id"]
        ].duplicated().sum()
    ]
})

final_key_validation

,dataset,duplicate_key_records
0,customers_analysis,0
1,sellers_analysis,0
2,products_analysis,0
3,geolocation_zip,0
4,orders_clean,0
5,order_items_clean,0
6,payments_clean,0
7,reviews_clean,0


### 8.2 Required Field Validation

Key components, core identifiers, foreign keys, and essential analytical
fields are checked for missing values before the processed datasets are
exported.

Missing values that were intentionally preserved during cleaning, such as
optional review comments, product measurements, lifecycle timestamps, and
unmatched geographic coordinates, are not treated as validation failures.

In [52]:
required_field_validation = pd.DataFrame({
    "dataset": [
        "customers_analysis",
        "customers_analysis",
        "sellers_analysis",
        "products_analysis",
        "products_analysis",
        "geolocation_zip",
        "orders_clean",
        "orders_clean",
        "order_items_clean",
        "order_items_clean",
        "order_items_clean",
        "order_items_clean",
        "payments_clean",
        "payments_clean",
        "reviews_clean",
        "reviews_clean",
        "reviews_clean"
    ],
    "field": [
        "customer_id",
        "customer_unique_id",
        "seller_id",
        "product_id",
        "product_category_name_english",
        "geolocation_zip_code_prefix",
        "order_id",
        "customer_id",
        "order_id",
        "order_item_id",
        "product_id",
        "seller_id",
        "order_id",
        "payment_sequential",
        "review_id",
        "order_id",
        "review_score"
    ],
    "missing_values": [
        customers_analysis["customer_id"].isna().sum(),
        customers_analysis["customer_unique_id"].isna().sum(),
        sellers_analysis["seller_id"].isna().sum(),
        products_analysis["product_id"].isna().sum(),
        products_analysis["product_category_name_english"].isna().sum(),
        geolocation_zip["geolocation_zip_code_prefix"].isna().sum(),
        orders_clean["order_id"].isna().sum(),
        orders_clean["customer_id"].isna().sum(),
        order_items_clean["order_id"].isna().sum(),
        order_items_clean["order_item_id"].isna().sum(),
        order_items_clean["product_id"].isna().sum(),
        order_items_clean["seller_id"].isna().sum(),
        payments_clean["order_id"].isna().sum(),
        payments_clean["payment_sequential"].isna().sum(),
        reviews_clean["review_id"].isna().sum(),
        reviews_clean["order_id"].isna().sum(),
        reviews_clean["review_score"].isna().sum()
    ]
})

required_field_validation

,dataset,field,missing_values
0,customers_analysis,customer_id,0
1,customers_analysis,customer_unique_id,0
2,sellers_analysis,seller_id,0
3,products_analysis,product_id,0
4,products_analysis,product_category_name_english,0
5,geolocation_zip,geolocation_zip_code_prefix,0
6,orders_clean,order_id,0
7,orders_clean,customer_id,0
8,order_items_clean,order_id,0
9,order_items_clean,order_item_id,0


#### Final Validation Findings

All analysis-ready datasets passed the final key-integrity validation, with
no duplicate primary or composite keys detected.

Required key components, core identifiers, foreign keys, and essential
analytical fields also contain no missing values. Together, these checks
confirm that the cleaning and enrichment process preserved the intended
relational structure and essential analytical information.

Missing values intentionally retained during cleaning, including optional
review content, product measurements, order lifecycle timestamps, and
unmatched geographic coordinates, remain available for metric-specific
handling in downstream analysis.

### 8.3 Export Processed Datasets

The validated analysis-ready datasets are exported to the
`data/processed` directory.

The exported files preserve the relational structure of the Olist data and
serve as the standardized data source for subsequent Python, SQL, and Power BI
analysis.

In [53]:
integer_cols = [
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty"
]

products_analysis[integer_cols] = (
    products_analysis[integer_cols]
    .astype("Int64")
)

products_analysis[integer_cols].dtypes

product_name_lenght           Int64
product_description_lenght    Int64
product_photos_qty            Int64
dtype: object

In [54]:
processed_path = Path("../data/processed")
processed_path.mkdir(parents=True, exist_ok=True)

export_tables = {
    "customers.csv": customers_analysis,
    "sellers.csv": sellers_analysis,
    "products.csv": products_analysis,
    "geolocation_zip.csv": geolocation_zip,
    "orders.csv": orders_clean,
    "order_items.csv": order_items_clean,
    "payments.csv": payments_clean,
    "reviews.csv": reviews_clean
}

for filename, dataframe in export_tables.items():
    dataframe.to_csv(
        processed_path / filename,
        index=False
    )

print(f"Exported {len(export_tables)} processed datasets.")

Exported 8 processed datasets.


In [55]:
export_validation = []

for filename, dataframe in export_tables.items():
    file_path = processed_path / filename

    exported_df = pd.read_csv(file_path)

    export_validation.append({
        "file": filename,
        "file_exists": file_path.exists(),
        "expected_rows": len(dataframe),
        "exported_rows": len(exported_df),
        "expected_columns": dataframe.shape[1],
        "exported_columns": exported_df.shape[1]
    })

export_validation = pd.DataFrame(export_validation)

export_validation["row_count_match"] = (
    export_validation["expected_rows"]
    == export_validation["exported_rows"]
)

export_validation["column_count_match"] = (
    export_validation["expected_columns"]
    == export_validation["exported_columns"]
)

export_validation

,file,file_exists,expected_rows,exported_rows,expected_columns,exported_columns,row_count_match,column_count_match
0,customers.csv,True,99441,99441,8,8,True,True
1,sellers.csv,True,3095,3095,7,7,True,True
2,products.csv,True,32951,32951,10,10,True,True
3,geolocation_zip.csv,True,19011,19011,4,4,True,True
4,orders.csv,True,99441,99441,8,8,True,True
5,order_items.csv,True,112650,112650,7,7,True,True
6,payments.csv,True,103886,103886,5,5,True,True
7,reviews.csv,True,99224,99224,7,7,True,True


#### Export Validation Findings

All eight processed datasets were successfully exported to the
`data/processed` directory.

Each exported CSV file was read back and validated against its corresponding
in-memory analysis-ready dataset. All files contain the expected number of
rows and columns, with no discrepancies detected during export.

The processed datasets are therefore ready for downstream Python, SQL, and
Power BI analysis.

## 9. Data Cleaning Summary

The Olist datasets were cleaned and prepared for downstream analysis while
preserving valid transactional records and the relational structure of the
original data.

Key transformations completed in this notebook include:

- Converted ZIP code identifiers to string values and transactional date
  fields to datetime types.
- Removed 261,831 exact duplicate geolocation observations.
- Excluded 25 geographic observations outside the defined coordinate
  sanity-check bounds.
- Aggregated valid geolocation observations into 19,011 unique ZIP-level
  representative coordinates using median latitude and longitude.
- Enriched customer and seller datasets with geographic coordinates,
  achieving 99.72% and 99.77% geographic coverage, respectively.
- Reclassified four zero product-weight values as missing while preserving
  all product and transaction records.
- Assigned the explicit `unknown` category to products without usable
  category information and incorporated the provided English category
  translation lookup.
- Preserved unusual payment records, optional missing review content, and
  structurally missing order lifecycle timestamps rather than applying
  unsupported deletion or imputation.
- Preserved the original analytical grain and key integrity of the orders,
  order items, payments, and reviews datasets.
- Prepared separate analysis-ready relational tables instead of creating a
  fully flattened master table, reducing the risk of row multiplication and
  double-counting.
- Successfully exported and validated eight processed datasets for
  downstream analysis.

The resulting processed data provides a consistent and reproducible
foundation for sales, customer, delivery, SQL, and Power BI analysis.